# vitalPeriodic

"Periodic" data refers to data which is consistently interfaced from bedside vital signs monitors into eCareManager. Data are generally interfaced as 1 minute averages, and archived into the vitalPeriodic table as 5 minute median values.

The vital signs are **not** validated by care staff, so data quality can vary, but the use of 1 minute averages followed by 5 minute medians before archival removes many spurious readings.

In [ ]:
from pathlib import Path

import holoviews as hv
import hvplot.pandas  # noqa+
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt

from loguru import logger

from eicu_db_connection import DatabaseConnection

import numpy as np
import palettable
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
import seaborn as sns

from tqdm.auto import tqdm
import panel as pn

tqdm.pandas()

hv.extension("bokeh")

%matplotlib inline

In [ ]:
icefire = sns.color_palette("icefire", as_cmap=True)

In [ ]:
INTER_DATA_PATH = "../intermediary_data/"
print(f"Loading and saving data from/to {Path(INTER_DATA_PATH).resolve()}")

In [ ]:
connection = DatabaseConnection()
engine = connection.engine
query_schema = connection.query_schema

In [ ]:
conn = engine.connect().execution_options(
    stream_results=True)

In [ ]:
df_apache_valid = pd.read_csv(filepath_or_buffer="../intermediary_data/apache_aps_var_results_valid_mortality.csv")
df_apache_valid = df_apache_valid.drop(columns=["Unnamed: 0", "Unnamed: 0_x", "Unnamed: 0_y"])
df_apache_valid.head(n=3)

# Getting the vitalperiodic data for patients with valid apache mortality prediction

In [ ]:
patient_stays_valid: list = df_apache_valid["patientunitstayid"].tolist()

### Change the column of interest here

In [ ]:
column_of_interest = "cvp"

Those are the patientunitstayids with a valid apache score (predictedmortality != -1)

In [ ]:
query_valid_apache = query_schema + f"""
SELECT {column_of_interest}
FROM vitalperiodic
WHERE patientunitstayid in ({", ".join(map(str, patient_stays_valid))}) 
ORDER BY patientunitstayid, observationoffset
"""

In [ ]:
vp_apache_path = Path(INTER_DATA_PATH) / Path("vital_periodic_with_apache_all.csv")

if vp_apache_path.is_file():
    df_vp_apache_valid = pd.read_csv(vp_apache_path,
                                     usecols=[column_of_interest, "patientunitstayid", "observationoffset"], engine="pyarrow")
    logger.info(f"Loading data from local csv.")
else:
    df_vp_apache_valid = pd.read_sql_query(query_valid_apache, engine)
    df_vp_apache_valid.to_csv(path_or_buf=vp_apache_path)
    logger.info(f"Loading data from database.")

In [ ]:
unique_stays = df_vp_apache_valid["patientunitstayid"].unique()
print(
    f"We got {len(unique_stays)} unique patientunitids with an Apache score and the vitalperiodic table in the dataset")

This number is different from the number of valid Apache Scores:

In [ ]:
print(f"{len(patient_stays_valid) - len(unique_stays)} patientunitstayids are missing in the vitalperiodic table")

### Time Series Heatmap of the Heartrate


In [ ]:
def convert_to_timedelta(stay):
    stay["time_delta"] = pd.to_timedelta(stay["observationoffset"] - stay["observationoffset"].iloc[0], unit="min")

    return stay

**Progress_apply is not native to pandas, its a tqdm extenstion**

In [ ]:
df_vp_apache_valid_grouped_time = df_vp_apache_valid.groupby("patientunitstayid").progress_apply(
    convert_to_timedelta).reset_index(drop=True)
# reset_index weil sonst die id zum indexer wird

In [ ]:
df_vp_apache_valid_grouped_time.head()

In [ ]:
def min_to_hours(x, pos):
    total_seconds = int(x / 1e9)
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60

    return f"{hours}:{minutes:02d}"


def five_min_to_hours(x, pos):
    total_seconds = 5 * 60 * int(x)
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60

    return f"{hours}:{minutes:02d}"


In [ ]:
total_time_pb = len(df_vp_apache_valid_grouped_time.groupby("patientunitstayid"))
total_time_pb

In [ ]:

#df_param_time_conc = pd.concat(investigated_parameter_df_list[0:100], axis=1, ignore_index=False)

In [ ]:
from notebooks_dani.plot_helpers import plot_vitalperiodic_ts_heatmap
from holoviews.operation.datashader import datashade, rasterize, shade, dynspread, spread

#fig_heatmesh, holo_heatmesh = plot_vitalperiodic_ts_heatmap(frame_with_time_delta=df_vp_apache_valid_grouped_time, variable_col="heartrate", #num_stays=50, plot_holo=True)

#### Interactive Heatmap

In [ ]:
#rast = hv.Layout(rasterize(holo_heatmesh))
#heatmesh = pn.Row(rast)
#heatmesh.show()

In [ ]:
variable_for_heatmap = column_of_interest
filename_corr = variable_for_heatmap + "_TS_heatmap"
notebook_name = "vitalperiodic_exploration"
plot_file_corr_path = Path("plot_outputs") / Path(notebook_name) / Path(filename_corr)

plot_vitalperiodic_ts_heatmap(frame_with_time_delta=df_vp_apache_valid_grouped_time, variable_col=variable_for_heatmap, num_stays=50, fig_width=30, fig_height=10, plot_holo=False)

plt.savefig(str(plot_file_corr_path) + ".png", dpi=300, bbox_inches="tight")